# <font color="#418FDE" size="6.5" uppercase>**Metriken und Suche**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Wählen passende Metriken für Klassifikation und Regression aus. 
- Führen leakage-sichere Kreuzvalidierung mit passenden Splittern durch. 
- Optimieren Hyperparameter mit kleinen Suchräumen und dokumentieren Ergebnisse. 


## **1. Metriken vertiefen**

### **1.1. Klassifikation bewerten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_01.jpg?v=1784800056" width="250">



>* Genauigkeit allein kann stark täuschen
>* Klassenverteilung und Fehlerfolgen immer prüfen

>* Precision, Recall und F1 zeigen Fehlerarten.
>* Wähle Metriken nach fachlichem Risiko.

>* Metrik nach Ziel und Risiko wählen
>* Fehler, Klassenverteilung und Einsatz transparent machen



In [ ]:
#@title Python-Code - Klassifikation bewerten

# Dieses Beispiel bewertet ein Klassifikationsmodell.
# Wir vergleichen Accuracy, Precision, Recall und F1.
# Die Ausgabe zeigt unterschiedliche Metrik-Perspektiven.

import sklearn
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

# Ein unausgewogener Datensatz macht Accuracy besonders trügerisch.
features, target = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    weights=[0.95, 0.05],
    random_state=42,
)

# Die Aufteilung bleibt durch Stratifikation fair verteilt.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Diese Prüfung schützt vor einer ungeeigneten Testmenge.
positive_count = int(sum(y_test))
if positive_count == 0:
    raise ValueError("Die Testmenge enthält keine positiven Fälle.")

# Das Dummy-Modell sagt immer die häufigste Klasse voraus.
model = DummyClassifier(strategy="most_frequent", random_state=42)
model.fit(X_train, y_train)

# Jetzt bewerten wir dieselben Vorhersagen aus mehreren Blickwinkeln.
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)

# Die Ausgabe bleibt kurz und fokussiert auf die Metriken.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Positive Fälle im Test: {positive_count} von {len(y_test)}")
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Wert: {f1:.3f}")
print("Merke: Hohe Accuracy kann wichtige positive Fälle übersehen.")



### **1.2. ROC und Precision Recall**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_02.jpg?v=1784800058" width="250">



>* Modelle über viele Schwellen vergleichen
>* Zielkonflikt zwischen Treffern und Fehlalarmen erkennen

>* ROC zeigt Trennung über viele Schwellen.
>* Bei Ungleichgewicht kann ROC zu optimistisch wirken.

>* Precision-Recall hilft bei seltenen positiven Fällen
>* Metrikwahl hängt von Kosten und Kontext ab



In [ ]:
#@title Python-Code - ROC und Precision Recall

# Wir vergleichen ROC und Precision-Recall anschaulich.
# Seltene positive Fälle verändern die Metrikbewertung.
# Die Kurven zeigen unterschiedliche Modellperspektiven.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

# Ein unausgeglichenes Klassifikationsproblem macht Precision-Recall wichtig.
features, target = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=4,
    n_redundant=0,
    weights=[0.92, 0.08],
    class_sep=1.0,
    random_state=42,
)

# Eine einfache logistische Regression liefert Wahrscheinlichkeiten.
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(features, target)
positive_scores = model.predict_proba(features)[:, 1]

# Die Zielvariable muss genau zwei Klassen enthalten.
unique_classes = np.unique(target)
if len(unique_classes) != 2:
    raise ValueError("Dieses Beispiel braucht genau zwei Klassen.")

# ROC betrachtet Trefferquote gegen Fehlalarmquote.
fpr, tpr, roc_thresholds = roc_curve(target, positive_scores)
roc_auc = roc_auc_score(target, positive_scores)

# Precision-Recall fokussiert die Qualität positiver Vorhersagen.
precision, recall, pr_thresholds = precision_recall_curve(target, positive_scores)
pr_auc = auc(recall, precision)
positive_rate = target.mean()

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Anteil positiver Fälle: {positive_rate:.2f}")
print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")

# Beide Kurven nutzen dieselben Scores, betonen aber andere Fragen.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, label=f"ROC-Kurve, AUC={roc_auc:.3f}")
ax.plot(recall, precision, label=f"Precision-Recall, AUC={pr_auc:.3f}")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="ROC-Zufallslinie")
ax.axhline(positive_rate, linestyle=":", color="black", label="PR-Basisrate")
ax.set_title("ROC und Precision-Recall bei seltenen positiven Fällen")
ax.set_xlabel("ROC: Fehlalarmquote | PR: Recall")

ax.set_ylabel("ROC: Trefferquote | PR: Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")

plt.show()



### **1.3. Regressionsfehler verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_01_03.jpg?v=1784800060" width="250">



>* Regression sagt Zahlenwerte statt Klassen voraus
>* Metriken müssen zum Anwendungskontext passen

>* MAE zeigt typische Abweichungen verständlich.
>* MSE/RMSE betonen große Fehler und Ausreißer.

>* Fehler immer mit Referenzmodell vergleichen
>* Mehrere passende Metriken vorher festlegen



In [ ]:
#@title Python-Code - Regressionsfehler verstehen

# Dieses Beispiel vergleicht wichtige Regressionsfehler.
# Ausreißer zeigen unterschiedliche Metrikreaktionen.
# Die Ausgabe hilft bei der Metrikwahl.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error

# Kleine Beispieldaten halten die Rechnung gut nachvollziehbar.
true_values = np.array([100, 120, 130, 150, 170, 190], dtype=float)
predictions_good = np.array([105, 118, 128, 152, 168, 188], dtype=float)
predictions_outlier = np.array([105, 118, 128, 152, 168, 140], dtype=float)

# Diese Prüfung verhindert unpassende Vergleichslängen.
if true_values.shape != predictions_good.shape:
    raise ValueError("Wahre Werte und Vorhersagen müssen gleich lang sein.")

# Dieselbe Prüfung gilt für das zweite Vorhersageszenario.
if true_values.shape != predictions_outlier.shape:
    raise ValueError("Alle Vorhersagen müssen gleich lang sein.")

# Drei Metriken zeigen verschiedene fachliche Blickwinkel.
mae_good = mean_absolute_error(true_values, predictions_good)
rmse_good = mean_squared_error(true_values, predictions_good) ** 0.5
medae_good = median_absolute_error(true_values, predictions_good)

# Ein einzelner großer Fehler verändert besonders RMSE.
mae_outlier = mean_absolute_error(true_values, predictions_outlier)
rmse_outlier = mean_squared_error(true_values, predictions_outlier) ** 0.5
medae_outlier = median_absolute_error(true_values, predictions_outlier)

print("Metriken für Vorhersagen in Euro:")
print(f"Ohne großen Ausreißer: MAE={mae_good:.1f}, RMSE={rmse_good:.1f}, MedAE={medae_good:.1f}")
print(f"Mit großem Ausreißer: MAE={mae_outlier:.1f}, RMSE={rmse_outlier:.1f}, MedAE={medae_outlier:.1f}")
print("Merke: RMSE bestraft große Fehler stärker als MAE und MedAE.")

# Das Diagramm macht den Ausreißer visuell sichtbar.
sample_numbers = np.arange(1, len(true_values) + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_numbers, true_values, marker="o", label="Tatsächlicher Wert")
ax.plot(sample_numbers, predictions_good, marker="o", label="Vorhersage ohne Ausreißer")
ax.plot(sample_numbers, predictions_outlier, marker="o", label="Vorhersage mit Ausreißer")

ax.set_title("Regressionsfehler und Ausreißer")
ax.set_xlabel("Beobachtung")
ax.set_ylabel("Wert in Euro")
ax.legend()
plt.show()



## **2. Leakage sichere Kreuzvalidierung**

### **2.1. KFold Varianten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_01.jpg?v=1784800062" width="250">



>* KFold schätzt Modellleistung über mehrere Folds
>* Vorverarbeitung nur im Trainingsfold lernen

>* KFold passt bei unabhängigen, ähnlichen Daten
>* Stratified KFold schützt vor unrepräsentativen Klassenfolds

>* Mischen verhindert verzerrte Folds durch Sortierung
>* Repeated KFold robuster, aber rechenintensiver



In [ ]:
#@title Python-Code - KFold Varianten

# Dieses Beispiel vergleicht wichtige KFold Varianten.
# Stratifikation schützt Klassenanteile in jedem Validierungsfold.
# Die Ausgabe zeigt leakage sichere Pipeline Bewertungen.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen absichtlich unausgeglichene Klassifikationsdaten.
features, target = make_classification(
    n_samples=120,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    weights=[0.9, 0.1],
    random_state=42,
)

# Diese Prüfung macht die Annahme des Beispiels sichtbar.
positive_count = int(np.sum(target == 1))
if positive_count < 5:
    raise ValueError("Zu wenige positive Beispiele für fünf Folds.")

# Die Pipeline lernt Skalierung nur im jeweiligen Trainingsfold.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)

# KFold mischt zufällig, beachtet aber keine Klassenanteile.
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_scores = cross_val_score(model, features, target, cv=kfold, scoring="f1")

# StratifiedKFold hält die Klassenverteilung pro Fold ähnlicher.
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(
    model,
    features,
    target,
    cv=stratified,
    scoring="f1",
)

# Wir zählen positive Validierungsbeispiele pro Fold.
kfold_counts = []
for train_index, valid_index in kfold.split(features):
    kfold_counts.append(int(np.sum(target[valid_index] == 1)))

# Bei StratifiedKFold werden die Zielwerte beim Splitten übergeben.
stratified_counts = []
for train_index, valid_index in stratified.split(features, target):
    stratified_counts.append(int(np.sum(target[valid_index] == 1)))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Positive Klasse insgesamt: {positive_count} von {len(target)}")
print(f"KFold positive Validierungsfälle: {kfold_counts}")
print(f"StratifiedKFold positive Validierungsfälle: {stratified_counts}")
print(f"KFold F1 Mittelwert: {np.mean(kfold_scores):.3f}")
print(f"StratifiedKFold F1 Mittelwert: {np.mean(stratified_scores):.3f}")



### **2.2. Gruppen und Zeit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_02.jpg?v=1784800065" width="250">



>* Gruppen nie über Training und Validierung mischen
>* So misst du echte Generalisierung auf neue Gruppen

>* Zeitliche Reihenfolge verhindert Zukunfts-Leakage
>* Historisch trainieren, später realistisch validieren

>* Gruppen und Zeit gemeinsam berücksichtigen
>* Splitter nach späterem Einsatz wählen



In [ ]:
#@title Python-Code - Gruppen und Zeit

# Wir vergleichen zufällige, gruppierte und zeitliche Kreuzvalidierung.
# Gruppen und Zeit verhindern typische Leakage-Fehler.
# Die Ausgaben zeigen realistischere Validierungswerte.

import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_val_score

# Ein kleiner Datensatz simuliert Messungen vieler Kundinnen.
rng = np.random.default_rng(42)
n_groups = 30
rows_per_group = 4

# Jede Gruppe hat einen eigenen Grundwert und mehrere Beobachtungen.
groups = np.repeat(np.arange(n_groups), rows_per_group)
group_signal = rng.normal(0, 1, n_groups)
time_index = np.tile(np.arange(rows_per_group), n_groups)

# Das Ziel hängt stark von der Gruppe und leicht von der Zeit ab.
noise = rng.normal(0, 0.35, groups.size)
score = group_signal[groups] + 0.15 * time_index + noise
y = (score > 0).astype(int)

# Die Gruppenkennung als Merkmal macht zufällige Folds zu optimistisch.
X = np.column_stack((groups, time_index))
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Ziel haben unterschiedlich viele Zeilen.")

# Ein einfaches Modell reicht für den Vergleich der Splitter.
model = LogisticRegression(max_iter=200, random_state=42)
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
group_cv = GroupKFold(n_splits=5)
time_cv = TimeSeriesSplit(n_splits=5)

# Zufällige Folds erlauben dieselben Gruppen in Training und Validierung.
random_scores = cross_val_score(model, X, y, cv=random_cv, scoring="accuracy")
group_scores = cross_val_score(
    model, X, y, cv=group_cv, groups=groups, scoring="accuracy"
)
time_scores = cross_val_score(model, X, y, cv=time_cv, scoring="accuracy")

# Die Mittelwerte zeigen, wie die Splitter die Fragestellung ändern.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Zufällige CV, mögliche Gruppen-Leakage: {random_scores.mean():.2f}")
print(f"GroupKFold, neue Gruppen geprüft: {group_scores.mean():.2f}")
print(f"TimeSeriesSplit, spätere Zeilen geprüft: {time_scores.mean():.2f}")
print("Merke: Der Splitter muss zur späteren Anwendung passen.")



### **2.3. cross validate anwenden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_02_03.jpg?v=1784800064" width="250">



>* Mehrfache Falten liefern stabilere Leistungsbewertung
>* Pipelines verhindern Leakage in jeder Falte

>* Mehrere Metriken zeigen Modellqualität umfassender.
>* Faltenstreuung und Laufzeiten bewerten Verlässlichkeit.

>* Passenden Splitter zur Datenstruktur wählen
>* Leakage vermeiden und Ergebnisse dokumentieren



In [ ]:
#@title Python-Code - cross validate anwenden

# Dieses Beispiel zeigt leakage-sichere Kreuzvalidierung.
# Pipeline lernt Skalierung nur innerhalb jeder Falte.
# cross_validate liefert mehrere stabile Bewertungswerte.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung macht die Annahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte haben unterschiedlich viele Zeilen.")

# Die Pipeline verhindert Leakage bei der Skalierung.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# StratifiedKFold erhält die Klassenverteilung in jeder Falte.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_validate berechnet mehrere Metriken pro Validierungsfalte.
scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={"accuracy": "accuracy", "recall": "recall"}
)

# Mittelwert und Streuung zeigen Leistung und Stabilität.
accuracy_mean = scores["test_accuracy"].mean()
accuracy_std = scores["test_accuracy"].std()
recall_mean = scores["test_recall"].mean()

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Datensatz: {X.shape[0]} Zeilen, {X.shape[1]} Merkmale")
print(f"Accuracy über 5 Falten: {accuracy_mean:.3f} ± {accuracy_std:.3f}")
print(f"Recall über 5 Falten: {recall_mean:.3f}")



## **3. Hyperparameter gezielt suchen**

### **3.1. Rastersuche mit Kreuzvalidierung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_01.jpg?v=1784800054" width="250">



>* Hyperparameter-Kombinationen systematisch und überschaubar testen
>* Kreuzvalidierung bewertet robuste Einstellungen zuverlässiger

>* Kleinen, fachlich sinnvollen Suchraum wählen
>* Datenlernende Schritte in Kreuzvalidierung einbetten

>* Mittelwert, Streuung und Metrik gemeinsam prüfen
>* Bestes Modell neu trainieren und testen



In [ ]:
#@title Python-Code - Rastersuche mit Kreuzvalidierung

# Diese Übung zeigt gezielte Rastersuche.
# Kreuzvalidierung bewertet mehrere Hyperparameter fair.
# Am Ende vergleichen wir dokumentierte Ergebnisse.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Pipeline verhindert Datenleckage bei der Skalierung.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", KNeighborsClassifier())]
)

# Der Suchraum bleibt bewusst klein und gut interpretierbar.
param_grid = {"model__n_neighbors": [3, 5, 9], "model__weights": ["uniform", "distance"]}

# StratifiedKFold erhält die Klassenanteile in jedem Fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV testet jede Kombination mit derselben Metrik.
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="recall", return_train_score=False
)

# Ein einziger Fit startet die komplette Rastersuche.
grid_search.fit(X, y)

# Wir dokumentieren die wichtigsten Ergebnisse kompakt.
results = pd.DataFrame(grid_search.cv_results_)
summary = results[
    ["param_model__n_neighbors", "param_model__weights", "mean_test_score", "std_test_score"]
]

# Die besten Zeilen zeigen Mittelwert und Stabilität.
summary = summary.sort_values("mean_test_score", ascending=False).head(3)
summary = summary.round({"mean_test_score": 3, "std_test_score": 3})

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Beste Parameter: {grid_search.best_params_}")
print(f"Bester mittlerer Recall: {grid_search.best_score_:.3f}")
print("Top-3-Kombinationen nach Kreuzvalidierung:")
print(summary.to_string(index=False))



### **3.2. Zufällige Suche**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_02.jpg?v=1784800050" width="250">



>* Zufällige Kombinationen sparen Rechenaufwand.
>* Wichtige Hyperparameter werden schneller gefunden.

>* Begründete Hyperparameterbereiche passend zum Parametertyp wählen
>* Alle Konfigurationen gleich kreuzvalidiert vergleichen

>* Suchprozess und Auswahlkriterien sauber dokumentieren
>* Beste Ergebnisse kritisch und zielbezogen bewerten



In [ ]:
#@title Python-Code - Zufällige Suche

# Wir vergleichen zufällige Hyperparameter-Kombinationen.
# Kreuzvalidierung bewertet jede Kombination fair.
# Die Ausgabe dokumentiert die beste Einstellung.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Ein kleines Klassifikationsproblem bleibt übersichtlich.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Pipeline verhindert Datenleckage beim Skalieren.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", KNeighborsClassifier())]
)

# Der Suchraum enthält wenige, gut erklärbare Optionen.
param_distributions = {
    "model__n_neighbors": np.arange(3, 22),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

# Derselbe Splitter bewertet alle gezogenen Kombinationen.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# RandomizedSearchCV zieht nur acht Kombinationen aus dem Suchraum.
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="balanced_accuracy",
    cv=cv,
    random_state=42,
)

# Das Training umfasst Suche und Kreuzvalidierung.
search.fit(X, y)

# Eine kleine Tabelle dokumentiert die wichtigsten Ergebnisse.
results = pd.DataFrame(search.cv_results_)
summary = results[
    ["param_model__n_neighbors", "param_model__weights", "param_model__p", "mean_test_score"]
]

# Die besten drei Konfigurationen werden kompakt angezeigt.
summary = summary.sort_values("mean_test_score", ascending=False).head(3)
summary = summary.rename(
    columns={"mean_test_score": "balanced_accuracy"}
)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Getestete Zufallskombinationen: {search.n_iter}")
print(f"Beste balanced accuracy: {search.best_score_:.3f}")
print("Beste drei dokumentierte Konfigurationen:")
print(summary.round(3).to_string(index=False))



### **3.3. Suchergebnisse dokumentieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_A/image_03_03.jpg?v=1784800052" width="250">



>* Geprüfte Einstellungen und Metriken nachvollziehbar festhalten
>* Stabilität und fachliche Passung bewerten

>* Ergebnisse mit Alternativen fachlich vergleichen
>* Einfachere, robuste Modelle begründet auswählen

>* Suchgrenzen und Auswahlgründe klar festhalten
>* Teststatus dokumentieren, Entscheidungen nachvollziehbar machen



In [ ]:
#@title Python-Code - Suchergebnisse dokumentieren

# Wir dokumentieren eine kleine Hyperparametersuche.
# Kreuzvalidierung liefert vergleichbare Suchergebnisse.
# Eine Tabelle begründet die Modellentscheidung.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Wir nutzen einen kleinen Klassifikationsdatensatz aus scikit-learn.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Fehlermeldungen später.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Pipeline skaliert innerhalb jeder Kreuzvalidierungsfaltung leakage-sicher.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Der Suchraum bleibt klein und gut dokumentierbar.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV speichert Mittelwert und Streuung jeder geprüften Variante.
search = GridSearchCV(
    pipeline, param_grid, scoring="recall", cv=cv, return_train_score=False
)
search.fit(X, y)

# Wir bauen eine kurze Ergebnisdokumentation statt nur den Sieger zu nennen.
results = search.cv_results_
summary = []

for index in range(len(results["params"])):
    row = {
        "C": results["param_model__C"][index],
        "mean_recall": round(results["mean_test_score"][index], 3),
        "std_recall": round(results["std_test_score"][index], 3),
        "rank": results["rank_test_score"][index],
    }
    summary.append(row)

summary = sorted(summary, key=lambda row: (row["rank"], row["C"]))
best = summary[0]

print(f"scikit-learn-Version: {sklearn.__version__}")
print("Dokumentierte Suche: Metrik=Recall, CV=5-fach stratifiziert")
print("Rang | C | mittlerer Recall | Streuung")

for row in summary:
    text = f"{row['rank']} | {row['C']} | {row['mean_recall']} | {row['std_recall']}"
    print(text)

print(f"Auswahl: C={best['C']} mit Recall={best['mean_recall']}.")



# <font color="#418FDE" size="6.5" uppercase>**Metriken und Suche**</font>


In this lecture, you learned to:
- Wählen passende Metriken für Klassifikation und Regression aus. 
- Führen leakage-sichere Kreuzvalidierung mit passenden Splittern durch. 
- Optimieren Hyperparameter mit kleinen Suchräumen und dokumentieren Ergebnisse. 

In the next Lecture (Lecture B), we will go over 'Merkmale erklären'